# 🧬 Genome-Doc — DGI Training (Colab)

This notebook trains the **Document Genome Inferrer (DGI)** module.
DGI reads degraded document images and outputs a structured Genome JSON
containing text, bounding boxes, layout, and style information.

**Architecture:** Donut (Swin Transformer encoder + BART decoder) with LoRA adapters + Layout Head

**Prerequisite:** SIR module must already be trained (we don't need the SIR checkpoint for DGI, but the dataset must be ready).

**Runtime:** Set to **GPU** (Runtime → Change runtime type)

## 1️⃣ Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2️⃣ Mount Drive & Extract Project

Same two-zip method as SIR training:
1. genome-doc.zip — Full project + 30K dataset
2. genome-cleaned.zip — Updated code files only

In [ ]:
# === STEP 1: Extract BASE Project + Dataset ===
from google.colab import drive
drive.mount('/content/drive')

!unzip -o /content/drive/MyDrive/PROMETHEUS/genome-doc.zip -d /content/

!echo "\n✅ Base project + dataset extracted!\n"
!ls /content/genome-doc/

In [ ]:
# === STEP 2: Extract UPDATED Code ===
!unzip -o /content/drive/MyDrive/PROMETHEUS/genome-cleaned.zip -d /content/

!echo "\n✅ Codebase updated with latest files!\n"
!ls /content/genome-doc/

In [ ]:
# Set working directory
import os
os.chdir('/content/genome-doc')
!pwd
!ls

## 3️⃣ Install Dependencies

DGI requires HuggingFace transformers, peft (for LoRA), and sentencepiece (for the Donut tokenizer).

In [ ]:
!pip install pyyaml pillow numpy tqdm pydantic
!pip install torch torchvision --upgrade
!pip install transformers accelerate peft sentencepiece
print("\n✅ All DGI dependencies installed!")

## 4️⃣ Verify Dataset

In [ ]:
import os

# Check train set
train_clean = 'data/synthetic/train/images/clean'
train_genomes = 'data/synthetic/train/genomes'
val_clean = 'data/synthetic/val/images/clean'
val_genomes = 'data/synthetic/val/genomes'

for label, path in [('Train Clean', train_clean), ('Train Genomes', train_genomes),
                     ('Val Clean', val_clean), ('Val Genomes', val_genomes)]:
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f"  ✅ {label}: {count} files")
    else:
        print(f"  ❌ {label}: NOT FOUND at {path}")

## 4.5️⃣ Split Train/Val (if not already done)

Run this ONLY if your val directory is empty or missing.

In [ ]:
import shutil
import random

train_clean = 'data/synthetic/train/images/clean'
train_genomes = 'data/synthetic/train/genomes'
val_clean = 'data/synthetic/val/images/clean'
val_genomes = 'data/synthetic/val/genomes'

# Only split if val doesn't exist yet
if not os.path.exists(val_clean) or len(os.listdir(val_clean)) == 0:
    os.makedirs(val_clean, exist_ok=True)
    os.makedirs(val_genomes, exist_ok=True)

    all_ids = [f.replace('.png', '') for f in os.listdir(train_clean) if f.endswith('.png')]
    random.seed(42)
    random.shuffle(all_ids)

    split_idx = int(len(all_ids) * 0.9)
    val_ids = all_ids[split_idx:]

    for sid in val_ids:
        shutil.move(f'{train_clean}/{sid}.png', f'{val_clean}/{sid}.png')
        if os.path.exists(f'{train_genomes}/{sid}.json'):
            shutil.move(f'{train_genomes}/{sid}.json', f'{val_genomes}/{sid}.json')

    print(f'✅ Split complete! Train: {split_idx}, Val: {len(val_ids)}')
else:
    print(f'✅ Val set already exists with {len(os.listdir(val_clean))} samples')

## 5️⃣ Update DGI Config for 96GB VRAM

In [ ]:
import yaml

config_path = 'configs/dgi_donut.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

config['training']['batch_size'] = 8
config['training']['gradient_accumulation'] = 4
config['data']['num_workers'] = 8

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('✅ DGI config updated for 96GB VRAM!')
print(f"  batch_size: {config['training']['batch_size']}")
print(f"  gradient_accumulation: {config['training']['gradient_accumulation']}")
print(f"  effective_batch: {config['training']['batch_size'] * config['training']['gradient_accumulation']}")
print(f"  num_workers: {config['data']['num_workers']}")

## 6️⃣ DGI Training

The DGI model will:
1. Download the pretrained Donut backbone (~700MB, one-time)
2. Apply LoRA adapters to the decoder attention layers
3. Train with combined Cross-Entropy + BBox L1 + GIoU loss

**Estimated time on 96GB Blackwell (batch=8, effective=32):** ~2-4 hours for 30 epochs

In [ ]:
# Full DGI training
!python training/train_dgi.py \
    --config configs/dgi_donut.yaml \
    --data-dir data/synthetic/train \
    --output-dir checkpoints/dgi

## 7️⃣ Export Checkpoints to Google Drive

In [ ]:
import shutil
import json
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
drive_dir = f'/content/drive/MyDrive/PROMETHEUS/dgi_training_{timestamp}'
os.makedirs(drive_dir, exist_ok=True)
print(f'📁 Saving to: {drive_dir}')

# Copy checkpoints
checkpoint_dir = 'checkpoints/dgi'
if os.path.exists(checkpoint_dir):
    for f in os.listdir(checkpoint_dir):
        src = os.path.join(checkpoint_dir, f)
        dst = os.path.join(drive_dir, f)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(src) / (1024 * 1024)
            print(f'  ✅ Saved: {f} ({size_mb:.1f} MB)')

# Save config
shutil.copy2('configs/dgi_donut.yaml', os.path.join(drive_dir, 'dgi_donut.yaml'))
print('  ✅ Saved: dgi_donut.yaml')

# Save training summary
summary = {
    'project': 'PROMETHEUS / Genome-Doc',
    'module': 'DGI (Document Genome Inferrer)',
    'timestamp': timestamp,
    'gpu': os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip(),
    'backbone': 'naver-clova-ix/donut-base',
    'batch_size': 8,
    'gradient_accumulation': 4,
    'effective_batch': 32,
    'learning_rate': 5e-5,
    'lora_rank': 16,
    'lora_alpha': 32,
}
with open(os.path.join(drive_dir, 'training_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('  ✅ Saved: training_summary.json')

print(f'\n🎉 Everything saved to Google Drive!')
print(f'📂 {drive_dir}')